In [79]:
import pandas as pd
import numpy as np
from statistics import multimode

# **Handling Duplicates**

Read-in data and duplicate IDs.

In [80]:
full_data = pd.read_csv("./asteroseismic_full.csv")
bad_ids = (pd.read_csv("./bad_ids.csv"))["bad_ids"].to_list()
good_ids = (pd.read_csv("./good_ids.csv"))["good_ids"].to_list()

Define weighted nanmean and EV state functions. Uncertainty in weighted mean taken as $\frac{\sum_i^nx_iw_i}{\sqrt{n}}$, where $n$ is the length of the valid data.

In [81]:
def weighted_nanmean(data, errors):

    # remove stars with unphysical errors (eg, err = 0)
    df = pd.DataFrame({"data": data, "errors": errors})
    df = df[df["errors"] != 0]
    # remove stars with nan measurements
    df = df.dropna()

    # convert to numpy arrays
    data = df["data"].to_numpy()
    errors = df["errors"].to_numpy()

    if len(errors) == 0:
        return np.nan, np.nan

    # calculate weights
    weights = 1 / (errors**2)
    assert len(weights) == len(data), "Weights and data must have equal length." # ensure dropna() operated correctly

    # calculate mean
    weighted_mean = np.sum(data * weights) / np.sum(weights)
    uncertainty_in_mean = np.std(data * weights) / np.sqrt(len(data))
    return weighted_mean, uncertainty_in_mean

def weighted_evstate(data):
    df = pd.DataFrame({"data": data}).dropna()

    # convert to numpy arrays
    data = df["data"].to_numpy()

    # return 0 for no determinations of EV state
    if len(data) == 0:
        return int(0)

    # find mode(s)
    modes = multimode(data)

    # when decisive EV state, use value
    if len(modes) == 1: 
        return int(modes[0])

    # no clear state, set to unknown
    elif len(modes) > 1: 
        return int(0)
    

Remove bad IDs.

In [82]:
full_data = full_data[~full_data["gaiadr3_id"].isin(bad_ids)]

In [83]:
results = []

for id in good_ids:
    # take a single duplicate ID data
    subset = full_data[full_data["gaiadr3_id"] == id]

    # take first value for identifiers -- should be consistent??
    assert len(subset["ra"].unique()) == 1, "Conflicting Right Ascensions for same Gaia DR3 ID."
    assert len(subset["dec"].unique()) == 1, "Confliction Declinations for same Gaia DR3 ID."
    row = {"gaiadr3_id": id, "ra": subset["ra"].iloc[0], "dec": subset["dec"].iloc[0]}

    # take weighted nanmeans for asteroseismic params
    for ast in ["numax", "dnu"]:
        mean, err = weighted_nanmean(subset[ast], subset[f"e_{ast}"])
        row[ast] = mean
        row[f"e_{ast}"] = err

    # determine likely evolutionary state
    row["ev_state"] = weighted_evstate(subset["ev_state"])

    # iterate through remaining columns 
    for survey, params in {
        "GAIA": ["teff", "logg", "fe_h"],
        "GSPSPEC": ["teff", "logg", "fe_h", "si_fe", "ca_fe", "ti_fe", "mg_fe", "nd_fe",
        "s_fe", "zr_fe", "n_fe", "cr_fe", "ce_fe", "ni_fe"]
    }.items():
        for param in params:
            # find column names
            data = f"{survey}-{param}"
            errors = f"{survey}-e_{param}"
            # find weighted means
            mean, err = weighted_nanmean(subset[data], subset[errors])
            # add to rows dictionary
            row[data] = mean
            row[errors] = err

    results.append(row)

good_dups = pd.DataFrame(results)

Write to `.csv` file.

In [84]:
good_dups.to_csv("./meaned_duplicates.csv", index=False)

Remove good IDs from `full_data` and add new means to new `.csv` file.

In [85]:
full_data = full_data[~full_data["gaiadr3_id"].isin(good_ids)]
assert full_data.columns.equals(good_dups.columns) # ensure equal columns
full_data = pd.concat([full_data, good_dups], ignore_index=True)
full_data.to_csv("./asteroseismic_clean.csv", index=False)